# HARIBON Objective 2 — Ensemble Walkthrough

This notebook is a readable version of the ensemble runner for **red tide risk prediction**. It keeps the ensemble logic unchanged while making the execution order easier to inspect:

1. load and prepare the dataset,
2. build the common 6-split rolling-origin evaluation set,
3. run per-model inference for LSTM, GRU, Transformer, and XGBoost,
4. apply the ensemble strategies,
5. aggregate metrics, and
6. compare the ensemble against the current Objective 2 baselines.

The evaluation focus is on accuracy, false-alarm reduction, and missing-data robustness. The ensemble is evaluated on the shared 6-split window used by the script so comparisons are consistent across the current Objective 2 benchmark. The recommended ensemble to use is `stacked`, because it has the highest AUC across the current 6-split benchmark.

## 1. Setup

Import the shared ensemble helpers and add the local `code/` directory to the path so the notebook reuses the same implementation as `run_ensemble.py`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

workspace_root = Path.cwd()
if (workspace_root / 'ensemble_model' / 'code').exists():
    ensemble_dir = workspace_root / 'ensemble_model'
elif (workspace_root / 'code').exists():
    ensemble_dir = workspace_root
else:
    raise FileNotFoundError('Could not locate ensemble_model/code from the current working directory.')

code_dir = ensemble_dir / 'code'
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

from ensemble_data import DEFAULT_DATASET_PATH, IMPUTATION_METHODS, build_splits, load_and_prepare
from ensemble_evaluate import compute_metrics, build_obj2_comparison, build_per_split_records, build_summary, print_obj2_comparison, print_summary_table, plot_ensemble_comparison
from ensemble_inference import predict_all
from ensemble_strategies import apply_all_strategies, stacked

## 2. Run Configuration

The defaults mirror the script runner: one imputation method at a time, the hybrid-adaptive Transformer scenario, the shared 6 common splits, and the three ensemble strategies. Changing these values alters the benchmark, but it does not change the underlying model implementations.

In [ ]:
dataset_path = DEFAULT_DATASET_PATH
imputation_method = 'hybrid_adaptive'
transformer_scenario = 'hybrid_adaptive'
selected_splits = [1, 2, 3, 4, 5, 6]
strategies = ['soft_vote', 'weighted_avg', 'stacked']

results_dir = ensemble_dir / 'results'
per_split_csv = results_dir / 'ensemble_per_split_metrics.csv'
summary_csv = results_dir / 'ensemble_summary.csv'
obj2_compare_csv = results_dir / 'obj2_model_comparison_final.csv'

print('Dataset           :', dataset_path)
print('Imputation method :', imputation_method)
print('Transformer case  :', transformer_scenario)
print('Selected splits   :', selected_splits)
print('Strategies        :', strategies)

## 3. Load Data and Build Splits

The ensemble uses the same rolling-origin split construction as the script. Each split is a common evaluation window so the base models and ensemble strategies are compared on aligned test periods.

In [ ]:
df = load_and_prepare(dataset_path, imputation_method=imputation_method)
all_splits = build_splits(df)
splits = [split for split in all_splits if split.split_num in selected_splits]

print(f'Loaded {len(df):,} rows')
print(f'Active splits: {len(splits)}')
for split in splits:
    print(
        f'Split {split.split_num}: train <= {split.train_end} | '
        f'test {split.test_start} to {split.test_end} | '
        f'n_train={split.X_seq_train.shape[0]:,} | '
        f'n_test={split.X_seq_test.shape[0]:,} | '
        f'bloom_rate={float(np.mean(split.y_test)):.3f}'
    )

## 4. Per-Model Inference

Each split is passed through the four base predictors. The ensemble does not retrain those models; it only loads their saved artifacts and collects probability outputs for comparison.

In [ ]:
all_probs = []
all_y_test = []

for split in splits:
    print(f'Running inference for split {split.split_num}...')
    probs = predict_all(split_data=split, transformer_scenario=transformer_scenario)
    all_probs.append(probs)
    all_y_test.append(split.y_test)

print(f'Collected probability outputs for {len(all_probs)} splits.')
print('Per split model keys:', sorted(all_probs[0].keys()) if all_probs else [])

## 5. Ensemble Strategies

The strategy logic stays the same as the script:

- `soft_vote` averages the available model probabilities.
- `weighted_avg` uses each model's split-level AUC as a weight.
- `stacked` trains the logistic meta-learner with leave-one-out splits so it can learn a calibrated blend without changing the base models.

In [ ]:
all_strategy_probs = []

for split_index, (split, probs) in enumerate(zip(splits, all_probs)):
    auc_weights = {}
    for model_name, model_probs in probs.items():
        metrics = compute_metrics(split.y_test, model_probs)
        auc_value = metrics['auc']
        auc_weights[model_name] = float(auc_value) if not np.isnan(auc_value) else 0.0

    stacked_probs = None
    if 'stacked' in strategies and len(splits) > 1:
        stacked_probs = stacked(
            target_split_idx=split_index,
            all_split_probs=all_probs,
            all_y_test=all_y_test,
        )

    strategy_probs = apply_all_strategies(
        probs_dict=probs,
        auc_weights=auc_weights,
        stacked_probs=stacked_probs if 'stacked' in strategies else None,
    )
    strategy_probs = {name: values for name, values in strategy_probs.items() if name in strategies}
    all_strategy_probs.append(strategy_probs)

print('Applied ensemble strategies to all active splits.')

## 6. Metric Aggregation

This step reproduces the same per-split CSVs and summary tables as the script. The final ranking is based on AUC first, then F1, then recall, which keeps the red tide prediction comparison aligned with the current repository outputs.

In [ ]:
split_results = []
for split, probs, strat_probs in zip(splits, all_probs, all_strategy_probs):
    split_results.append({
        'split_num': split.split_num,
        'train_end': split.train_end,
        'test_start': split.test_start,
        'test_end': split.test_end,
        'n_train': int(split.X_seq_train.shape[0]),
        'n_test': int(split.X_seq_test.shape[0]),
        'y_test': split.y_test,
        'model_probs': probs,
        'strategy_probs': strat_probs,
    })

per_split_df = build_per_split_records(split_results)
per_split_df['imputation_method'] = imputation_method
summary_df = build_summary(per_split_df)
summary_df['imputation_method'] = imputation_method
obj2_df = build_obj2_comparison(summary_df)
obj2_df['imputation_method'] = imputation_method

results_dir.mkdir(parents=True, exist_ok=True)
per_split_df.to_csv(per_split_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
obj2_df.to_csv(obj2_compare_csv, index=False)

print_summary_table(summary_df)
print()
print_obj2_comparison(obj2_df)

## 6.5. Generate Ensemble Comparison Plot

Generate the bar chart showing the "Ensemble Boost" by comparing individual models vs ensemble strategies.

In [ ]:
# Generate AUC comparison plot
plot_ensemble_comparison(summary_df, metric="auc_mean")

# Generate F1 comparison plot  
plot_ensemble_comparison(summary_df, metric="f1_mean")

## 7. Final Readout

The cells below surface the current best ensemble strategy and the Objective 2 comparison leader for this run. If you change the imputation method or split selection, the notebook will recompute the ranking from the same code path.

In [ ]:
best_ensemble = summary_df[summary_df['source'] == 'ensemble'].sort_values(
    ['auc_mean', 'f1_mean', 'recall_mean'],
    ascending=[False, False, False],
).iloc[0]
best_obj2 = obj2_df.iloc[0]

print(f"Best ensemble strategy: {best_ensemble['name']}")
print(f"Best ensemble AUC   : {best_ensemble['auc_mean']:.3f}")
print(f"Objective 2 leader  : {best_obj2['model']}")
print(f"Objective 2 AUC     : {best_obj2['auc_mean']:.3f}")
print('')
print('Interpretation: this is a red tide risk prediction comparison, not a retraining step.')
print('The ensemble is evaluated on the shared 6-split window to match the current Objective 2 setup.')
print('Recommended ensemble model: stacked')
print('Why: it is rank 1 by AUC in the current 6-split ensemble benchmark across all imputation methods.')